In [1]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

ModuleNotFoundError: No module named 'nltk'

# PRE-TRAITMENT AND BOW

In [ ]:
import nltk
import numpy
import sklearn
import re
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import SnowballStemmer,WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from collections import Counter

# nltk.download('wordnet')
# nltk.download('punkt_tab')


folder_path="./books/"
lemmatizer=WordNetLemmatizer()
docs=[]
titles=[]
stop_words= set(stopwords.words('english'))
txt_files= [file for file in Path("./books").rglob("*.txt") if file.is_file]
for file_path in txt_files:
    with open(file_path, 'r', encoding='utf8') as file:
        title=f"{file_path}".replace('books/', '')
        titles.append(title)
        text = file.read().lower()
        text=re.sub(r"[^a-z]"," ",text)
        tokens=word_tokenize(text)
        filtered_tokens=[word for word in tokens if word not in stop_words and len(word)>2]
        lemmatized_words=[lemmatizer.lemmatize(word) for word in filtered_tokens]
        new_lemmatized_words=' '.join(lemmatized_words)
        docs.append(new_lemmatized_words)
        counter = Counter(lemmatized_words)
        df=pd.DataFrame.from_dict({"count":counter}).reset_index()
        df.columns=["word", "count"]
        print(title)
        print(df)
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(counter)
        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title("Word Cloud")
        plt.show()   
      

        


# TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
vectorizer = tfidf_vectorizer.fit_transform(docs)   
matrix = vectorizer.toarray()
df_tfidf = pd.DataFrame(matrix.T, columns=titles, index=tfidf_vectorizer.get_feature_names_out())
(df_tfidf)

# LATENTE SEMANTIC ANALYSIS (LSA)

In [ ]:

from sklearn.decomposition import TruncatedSVD


n_topics = 3
lsa_model =TruncatedSVD(n_components=n_topics, random_state=42)
topic_matrix = lsa_model.fit_transform(matrix)
topic_matrix2=pd.DataFrame(topic_matrix, columns=[f"topic{i+1}" for i in range(n_topics)])

topic_matrix2.insert(0, 'book', titles)
topic_matrix2



# DISPLAY TOP 10 FREQUENCY WORD

In [ ]:
terms = tfidf_vectorizer.get_feature_names_out()
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i+1}:")
    top_10_terms = topic.argsort()[-10:][::-1]
    top_terms = [terms[j] for j in top_10_terms]
    print(", ".join(top_terms))

# GROUP BOOK BY THEIR DOMINANT TOPIC


In [ ]:
import numpy as np

dominant_topics = np.argmax(topic_matrix, axis=1)
books_by_topic = {}
for i, topic_index in enumerate(dominant_topics):
    topic_name = f"Topic {topic_index+1}"
    if topic_name not in books_by_topic:
        books_by_topic[topic_name] = []
    books_by_topic[topic_name].append(titles[i])

for topic, books in books_by_topic.items():
    print(f"{topic}: {books}")
    for book in books:
        print(f"  - {book}")
    print()

# DOCUMENT SIMILARITY

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


indices = pd.Series(topic_matrix2['book'])
indices[:5]
similarity = cosine_similarity(topic_matrix)

def best_recommended_books(title, n):

    if title not in indices.values:
        return "Title not found in the database."
    recommended_books = []
    idx = indices[indices == title].index[0] 
    score_series = pd.Series(similarity[idx]).sort_values(ascending = False)
    top_10_indices = list(score_series.iloc[1:n+1].index) 
    
    for i in top_10_indices:
        recommended_books.append(list(topic_matrix2['book'])[i])
        
    return recommended_books



best_recommended_books("books\\The Phase Rule and Its Applications.txt",5)
    
